# Классификация обращений в службу поддержки
**Задача:** многоклассовая классификация текстов (8 категорий).
**Стек:** TF-IDF, LogisticRegression, LinearSVC, RandomForest, PyTorch (MLP), MLflow.

In [ ]:
import re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import mlflow, mlflow.sklearn, joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    try: nltk.data.find(f"corpora/{pkg}")
    except LookupError: nltk.download(pkg, quiet=True)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("support_tickets_classification")
print("MLflow tracking:", mlflow.get_tracking_uri())

## Шаг 1. Загрузка данных

In [ ]:
df = pd.read_csv("all_tickets_processed_improved_v3.csv")

print("Размер датасета (строки, колонки):", df.shape)
print("Колонки:", df.columns.tolist())
print("\nПропуски в данных:\n", df.isnull().sum())

display(df.head())

## Шаг 2. Разведочный анализ (EDA)

In [ ]:
# 1. Распределение классов
class_counts = df["Topic_group"].value_counts()
print("Распределение классов:\n", class_counts)
print("\nДоли классов (%):\n", (class_counts / len(df) * 100).round(2))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=class_counts.values, y=class_counts.index, palette="viridis", ax=ax)
ax.set_title("Распределение классов (дисбаланс налицо)")
for i, v in enumerate(class_counts.values):
    ax.text(v + 2, i, str(v), va="center")
plt.tight_layout()
plt.show()

# 2. Длина текста
df["text_len"] = df["Document"].astype(str).str.split().str.len()
print("\nСтатистика длины текста (слов):\n", df["text_len"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["text_len"], bins=50, ax=axes[0], color="steelblue")
axes[0].set_title("Распределение длины текстов")
sns.boxplot(x="Topic_group", y="text_len", data=df, ax=axes[1])
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("Длина текста по классам")
plt.tight_layout()
plt.show()

# 3. Облако слов
text_all = " ".join(df["Document"].astype(str))
wc = WordCloud(width=1200, height=600, background_color="white",
               max_words=150, colormap="viridis").generate(text_all)
plt.figure(figsize=(14, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Облако слов по всем обращениям")
plt.show()

## Шаг 3. Предобработка текста

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess(text: str) -> str:
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return " ".join(lemmatizer.lemmatize(w) for w in tokens)

df["clean_text"] = df["Document"].apply(preprocess)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)
print("После предобработки:", df.shape)
display(df[["Document", "clean_text", "Topic_group"]].head(3))

## Шаг 4. Разбиение данных (Train / Val / Test)

In [ ]:
# Явно преобразуем в numpy-массивы, чтобы избежать ошибки с pyarrow
X = np.array(df["clean_text"].tolist())
y = np.array(df["Topic_group"].tolist())

# 80% train, 20% test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Из train выделяем 15% в validation (для ранней остановки MLP)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=42,
    stratify=y_train_full)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
class_names = sorted(np.unique(y))
print("Классы:", class_names, "| n_classes =", len(class_names))

## Шаг 5-6. Классические модели + Pipeline + MLflow

In [7]:
TFIDF_PARAMS = dict(max_features=5000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)

def make_pipeline(clf):
    return Pipeline([("tfidf", TfidfVectorizer(**TFIDF_PARAMS)), ("clf", clf)])

def evaluate(y_true, y_pred, name):
    return dict(
        model=name,
        accuracy=accuracy_score(y_true, y_pred),
        precision_macro=precision_score(y_true, y_pred, average="macro", zero_division=0),
        recall_macro=recall_score(y_true, y_pred, average="macro", zero_division=0),
        f1_macro=f1_score(y_true, y_pred, average="macro", zero_division=0),
        f1_weighted=f1_score(y_true, y_pred, average="weighted", zero_division=0),
    )

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "LinearSVC": LinearSVC(max_iter=3000, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
}

results, trained = [], {}

for name, clf in models.items():
    pipe = make_pipeline(clf)
    with mlflow.start_run(run_name=name):
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        m = evaluate(y_test, y_pred, name)
        
        mlflow.log_param("model", name)
        for k, v in TFIDF_PARAMS.items(): mlflow.log_param(f"tfidf_{k}", v)
        for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]:
            mlflow.log_metric(k, m[k])
            
        mlflow.sklearn.log_model(pipe, artifact_path=f"model_{name}")
        results.append(m); trained[name] = pipe
        print(f"{name}: acc={m['accuracy']:.4f} | f1_macro={m['f1_macro']:.4f}")

2026/11/24 22:22:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LogisticRegression: acc=0.8424 | f1_macro=0.8398


2026/11/24 22:22:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LinearSVC: acc=0.8410 | f1_macro=0.8381


2026/11/24 22:23:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the specific types you have reviewed and trust to the `trusted` argument; avoid passing everything repo

## Шаг 7. Нейросетевая модель MLP (PyTorch) + MLflow

In [ ]:
# Векторизация для MLP (fit только на train)
tfidf_nn = TfidfVectorizer(**TFIDF_PARAMS)
Xtr = tfidf_nn.fit_transform(X_train).toarray().astype(np.float32)
Xvl = tfidf_nn.transform(X_val).toarray().astype(np.float32)
Xts = tfidf_nn.transform(X_test).toarray().astype(np.float32)

label_to_idx = {c: i for i, c in enumerate(class_names)}
ytr = np.array([label_to_idx[c] for c in y_train], dtype=np.int64)
yvl = np.array([label_to_idx[c] for c in y_val], dtype=np.int64)
yts = np.array([label_to_idx[c] for c in y_test], dtype=np.int64)

class DS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, h=256, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(p),
            nn.Linear(h, h//2), nn.ReLU(), nn.Dropout(p),
            nn.Linear(h//2, n_classes))
    def forward(self, x): return self.net(x)

BATCH, H, P, LR, EPOCHS, PAT = 32, 256, 0.3, 1e-3, 60, 10
train_ld = DataLoader(DS(Xtr, ytr), batch_size=BATCH, shuffle=True)
val_ld   = DataLoader(DS(Xvl, yvl), batch_size=BATCH)
test_ld  = DataLoader(DS(Xts, yts), batch_size=BATCH)

mlp = MLP(Xtr.shape[1], len(class_names), H, P).to(device)
crit = nn.CrossEntropyLoss()
opt = torch.optim.Adam(mlp.parameters(), lr=LR)

def run_eval(loader):
    mlp.eval(); loss_sum, correct, n = 0.0, 0, 0
    preds, tgts = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = mlp(xb); loss_sum += crit(logits, yb).item() * xb.size(0)
            pr = logits.argmax(1); correct += (pr == yb).sum().item(); n += xb.size(0)
            preds.append(pr.cpu().numpy()); tgts.append(yb.cpu().numpy())
    return loss_sum/n, correct/n, np.concatenate(preds), np.concatenate(tgts)

with mlflow.start_run(run_name="MLP"):
    for k, v in dict(input_dim=Xtr.shape[1], hidden=H, dropout=P, lr=LR, batch_size=BATCH, epochs=EPOCHS).items():
        mlflow.log_param(k, v)

    best_vl, best_state, bad = float("inf"), None, 0
    for ep in range(1, EPOCHS+1):
        mlp.train()
        for xb, yb in train_ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(mlp(xb), yb); loss.backward(); opt.step()
        vl, va, _, _ = run_eval(val_ld)
        if ep % 10 == 0: print(f"ep{ep:3d} val_loss={vl:.4f} val_acc={va:.4f}")
        if vl < best_vl - 1e-5:
            best_vl, bad = vl, 0
            best_state = {k: v.detach().cpu().clone() for k, v in mlp.state_dict().items()}
        else:
            bad += 1
            if bad >= PAT: print(f"Early stop @ {ep}"); break

    mlp.load_state_dict(best_state)
    _, _, yp, yt = run_eval(test_ld)
    yp_lbl = np.array([class_names[i] for i in yp])
    yt_lbl = np.array([class_names[i] for i in yt])
    m_mlp = evaluate(yt_lbl, yp_lbl, "MLP")
    for k in ["accuracy","precision_macro","recall_macro","f1_macro","f1_weighted"]:
        mlflow.log_metric(k, m_mlp[k])
    torch.save(mlp.state_dict(), "mlp.pth"); mlflow.log_artifact("mlp.pth")
    joblib.dump(tfidf_nn, "tfidf_mlp.pkl"); mlflow.log_artifact("tfidf_mlp.pkl")
    results.append(m_mlp)
    print(f"MLP: acc={m_mlp['accuracy']:.4f} | f1_macro={m_mlp['f1_macro']:.4f}")

## Шаг 8. Оценка качества и Confusion Matrix

In [ ]:
results_df = pd.DataFrame(results).set_index("model").sort_values("f1_macro", ascending=False)
print("=== Сводная таблица метрик (test) ===")
display(results_df.round(4))

with mlflow.start_run(run_name="summary"):
    results_df.to_csv("metrics_summary.csv")
    mlflow.log_artifact("metrics_summary.csv")

best = results_df.index[0]
print(f"\nЛучшая модель: {best}")

yp_best = (np.array([class_names[i] for i in yp]) if best == "MLP" else trained[best].predict(X_test))
print("\nclassification_report:")
print(classification_report(y_test, yp_best, zero_division=0))

cm = confusion_matrix(y_test, yp_best, labels=class_names)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", xticks_rotation=45, values_format="d")
plt.title(f"Confusion matrix — {best}")
plt.tight_layout(); plt.savefig("cm_best.png", dpi=120, bbox_inches="tight"); plt.show()

with mlflow.start_run(run_name="cm_best"):
    mlflow.log_artifact("cm_best.png")

## Шаг 9. Выводы
**Лучшая модель.** По F1-macro на test лучшей оказалась модель, указанная в сводной таблице. Выбор обоснован тем, что при дисбалансе accuracy «раздувается» за счёт доминирующих классов, а F1-macro показывает реальное качество по каждой категории.

**Худшие классы.** По confusion matrix и classification_report хуже всего предсказываются редкие классы (`Storage`, `Administrative rights`, `Internal Project`) — им не хватает примеров. Частые путаницы — между смежными по смыслу категориями: `Access ↔ HR Support`, `Hardware ↔ Purchase`.

**Что улучшить.** (1) Больше данных для редких классов. (2) Эмбеддинги `sentence-transformers` вместо TF-IDF. (3) `GridSearchCV` по гиперпараметрам. (4) Вырезать юридический boilerplate regex'ом.

**Production.** Решение **частично пригодно**: рабочий baseline для автоматической маршрутизации «первой линии» с ручной проверкой низкой уверенности. Для полноценного прода нужны: больше разметки, мониторинг дрифта, A/B-тест на реальном трафике.

In [ ]:
# Создаем README.md
readme_text = [
    "# Классификация обращений в службу поддержки\n\n",
    "## Задача\n",
    "Многоклассовая классификация текста: по `Document` предсказать `Topic_group`.\n",
    "8 категорий: Hardware, Access, Miscellaneous, HR Support, Purchase, Internal Project, Storage, Administrative rights.\n\n",
    "## Датасет\n",
    "`all_tickets_processed_improved_v3.csv` — колонки `Document`, `Topic_group`. Классы несбалансированы.\n\n",
    "## Запуск\n",
    "1. Установить зависимости: `pip install -r requirements.txt`\n",
    "2. Запустить ноутбук: `jupyter notebook support_tickets.ipynb`\n",
    "3. Посмотреть результаты в MLflow: `mlflow ui --backend-store-uri sqlite:///mlflow.db`\n\n",
    "## Результаты\n",
    "См. сводную таблицу в ноутбуке.\n"
]

with open("README.md", "w", encoding="utf-8") as f:
    f.writelines(readme_text)

# Создаем requirements.txt
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("pandas\nnumpy\nscikit-learn\nmatplotlib\nseaborn\nnltk\nwordcloud\ntorch\nmlflow\njoblib\nipykernel\n")

# Создаем .gitignore
with open(".gitignore", "w", encoding="utf-8") as f:
    f.write("mlruns/\nmlflow.db\n.ipynb_checkpoints/\n__pycache__/\n*.pth\n*.pkl\nall_tickets_processed_improved_v3.csv\n")

print("Файлы README.md, requirements.txt и .gitignore успешно созданы в папке проекта!")